In [2]:
import os 
import torch 
import torch.nn as nn
import torchvision.models  as models 
import torchvision.transforms as transforms 
from PIL import Image 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"-->Execution target confirmed: {device}")
if torch.cuda.is_available(): 
    print(f"--> Cloud Hardware Profile: {torch.cuda.get_device_name(0)}") 
    
backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

modules = list(backbone.children())[:-2]
feature_extractor = nn.Sequential(*modules).to(device) 
feature_extractor.eval() 

print("--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully")

-->Execution target confirmed: cuda
--> Cloud Hardware Profile: Tesla T4
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 141MB/s]


--> Pre-trained ResNet-50 backbone loaded. Classification head pruned sucessfully


In [3]:
import torch.nn as nn 
import torch.nn.functional as F 

class SaliencyDecoder(nn.Module): 
    def __init__(self): 
        super(SaliencyDecoder, self).__init__() 
        self.conv1 = nn.Conv2d(2048, 512, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(512, 128, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(128, 1, kernel_size=1)
        
    def forward(self, x): 
        x = F.relu(self.conv1(x)) 
        x = F.interpolate(x, scale_factor=4, mode='bilinear', align_corners=False)
        x = F.relu(self.conv2(x))
        x = F.interpolate(x, scale_factor=8, mode='bilinear', align_corners=False) 
        saliency_map = torch.sigmoid(self.conv3(x))
        return saliency_map

decoder = SaliencyDecoder().to(device)
print("--> Saliency Decoder initialized and pushed to Tesla T4 GPU.")

--> Saliency Decoder initialized and pushed to Tesla T4 GPU.


In [5]:
from torch.utils.data import Dataset, DataLoader 

class SynthethicSaliencyDataset(Dataset): 
    def __init__(self, num_samples=16): 
        self.num_samples = num_samples 
    
    def __len__(self): 
        return self.num_samples 
    
    def __getitem__(self, idx): 
        synthethic_image = torch.rand(3, 224, 224) 
        synthethic_fixation = torch.rand(1, 224, 224) 
        
        return synthethic_image, synthethic_fixation

test_dataset = SynthethicSaliencyDataset(num_samples=8) 
test_loader = DataLoader(test_dataset, batch_size = 2, shuffle = True)

In [7]:
import torch 

def compute_covariance_manifold(features): 
    B, C, H, W = features.shape 
    N = H * W
    
    reshaped_feats = features.view(B, C, N) 
    mean_feats = reshaped_feats.mean(dim=2, keepdim=True)
    centered_feats = reshaped_feats - mean_feats
    sliced_feats = centered_feats[:, :64, :] 
    cov_matrices = torch.bmm(sliced_feats, sliced_feats.transpose(1,2)) / (N-1)
    eps = 1e-4 * torch.eye(64, device=features.device).unsqueeze(0).repeat(B, 1, 1)
    spd_matrices = cov_matrices + eps 
    
    return spd_matrices